# 05 — Presentation Figures

High-resolution publication-ready figures for the Group 11 CMU Heinz presentation (April 21, 2026).
Run all cells after the full pipeline has completed.

**Narrative arc:**
1. How arms control norms evolved in treaty language (anchor analysis)
2. How states' rhetoric tracks those norms (NLP scoring)
3. How behavior compares to rhetoric (gap index)
4. ATT compliance: what ATT parties actually do vs what they say

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Presentation style
plt.rcParams.update({
    'figure.dpi': 200,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

METRICS = Path('../output/metrics')
EMBED  = Path('../output/embeddings')
FIGURES = Path('../output/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

print('Figures will be saved to:', FIGURES)

## Figure 1 — Treaty Anchor Similarity Over Time (P5 vs NAC)

In [ ]:
from src.groups import get_group_members

anchor_df = pd.read_csv(EMBED / 'anchor_scores.csv') if (EMBED / 'anchor_scores.csv').exists() \
            else pd.read_csv(METRICS / 'anchor_scores.csv')

p5  = get_group_members('p5')
nac = get_group_members('nac')
nato = get_group_members('nato')

anchor_cols = [c for c in anchor_df.columns if c.endswith('_score')]

fig, axes = plt.subplots(1, len(anchor_cols), figsize=(16, 4), sharey=True)
if len(anchor_cols) == 1:
    axes = [axes]

groups_map = {'P5': p5, 'NAC': nac, 'NATO': nato}
colors_map = {'P5': '#d73027', 'NAC': '#4575b4', 'NATO': '#74add1'}

for ax, col in zip(axes, anchor_cols):
    treaty = col.replace('_score', '').upper()
    for gname, members in groups_map.items():
        sub = anchor_df[anchor_df['country_code'].isin(members)]
        ts = sub.groupby('year')[col].mean()
        ax.plot(ts.index, ts.values, label=gname, color=colors_map[gname], linewidth=2)
    ax.set_title(treaty)
    ax.set_xlabel('Year')

axes[0].set_ylabel('Cosine similarity to treaty anchor')
axes[0].legend()
fig.suptitle('Treaty Anchor Similarity Over Time by Country Group', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'pres_fig1_anchor_similarity.png', dpi=200, bbox_inches='tight')
plt.show()

## Figure 2 — Rhetoric Score by Regime Type (Box Plot)

In [ ]:
rhetoric_df = pd.read_csv(METRICS / 'rhetoric_scores.csv')

# Add regime type if available (from vdem in companion pipeline)
# Fallback: use country group membership as proxy
from src.groups import get_group_members

def assign_regime(iso3):
    if iso3 in get_group_members('p5') or iso3 in ['DEU','FRA','GBR','JPN','AUS','NZL','CAN','NOR','SWE','DNK','FIN','AUT','IRL']:
        return 'Liberal Democracy'
    elif iso3 in ['IND','BRA','MEX','COL','ARG','TUR','IDN','PHL','THA','ZAF','NGA']:
        return 'Electoral Democracy'
    elif iso3 in ['RUS','CHN','SAU','ARE','QAT','IRN','EGY','MMR','KAZ','BLR','PRK','SYR']:
        return 'Autocracy'
    else:
        return 'Other'

rhetoric_df['regime_type'] = rhetoric_df['country_code'].apply(assign_regime)

order = ['Liberal Democracy', 'Electoral Democracy', 'Autocracy', 'Other']
palette = {'Liberal Democracy': '#4575b4', 'Electoral Democracy': '#74add1',
           'Autocracy': '#d73027', 'Other': '#999999'}

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=rhetoric_df[rhetoric_df['regime_type'] != 'Other'],
    x='regime_type', y='rhetoric_score',
    order=[o for o in order if o != 'Other'],
    hue='regime_type',
    palette=palette, legend=False,
    ax=ax, width=0.5, linewidth=1.2
)
ax.set_xlabel('')
ax.set_ylabel('Composite Rhetoric Score (0–1)')
ax.set_title('Arms Control Rhetoric Score by Regime Type', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'pres_fig2_rhetoric_by_regime.png', dpi=200, bbox_inches='tight')
plt.show()

## Figure 3 — Rhetoric-Action Gap Scatter (Hero Slide)

In [ ]:
gap_df = pd.read_csv(METRICS / 'rhetoric_action_gap.csv')
# Use most recent year with good coverage
year = gap_df['year'].max()
snap = gap_df[gap_df['year'] == year].copy()

fig, ax = plt.subplots(figsize=(11, 8))
scatter = ax.scatter(
    snap['rhetoric_score'], snap['action_score'],
    s=70, alpha=0.75, edgecolors='white', linewidth=0.5,
    c=snap['gap'], cmap='RdBu_r', vmin=-0.5, vmax=0.5
)
plt.colorbar(scatter, ax=ax, label='Gap (rhetoric − action)', shrink=0.7)

# Diagonal
ax.plot([0, 1], [0, 1], 'k--', alpha=0.35, linewidth=1)

# Annotations
notable = ['USA','RUS','CHN','FRA','GBR','DEU','SAU','IND','AUT','NZL','MEX','ZAF','IRN','PAK']
for _, row in snap[snap['country_code'].isin(notable)].iterrows():
    ax.annotate(row['country_code'],
                xy=(row['rhetoric_score'], row['action_score']),
                xytext=(5, 3), textcoords='offset points',
                fontsize=8.5, fontweight='bold')

# Zone labels
ax.text(0.75, 0.15, 'Quiet\nGood Actors', fontsize=9, color='#4575b4', alpha=0.7, ha='center')
ax.text(0.15, 0.80, 'Hypocrites', fontsize=9, color='#d73027', alpha=0.7, ha='center')

ax.set_xlabel('Rhetoric Score  (words)', fontsize=12)
ax.set_ylabel('Action Score  (behavior)', fontsize=12)
ax.set_title(f'Words vs Deeds: Rhetoric-Action Gap ({year})', fontsize=14, fontweight='bold')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'pres_fig3_gap_scatter.png', dpi=200, bbox_inches='tight')
plt.show()

## Figure 4 — Key Terms Over Time

In [ ]:
term_df = pd.read_csv(METRICS / 'term_trajectories.csv')
print(term_df.columns.tolist())
term_df.head()

In [ ]:
if 'term' in term_df.columns and 'year' in term_df.columns and 'mean_tfidf' in term_df.columns:
    highlight_terms = ['humanitarian', 'deterrence', 'civilian', 'verification',
                       'disarmament', 'non-proliferation']
    term_sub = term_df[term_df['term'].isin(highlight_terms)]

    fig, ax = plt.subplots(figsize=(13, 5))
    for term, grp in term_sub.groupby('term'):
        grp = grp.sort_values('year')
        ax.plot(grp['year'], grp['mean_tfidf'], label=term, linewidth=2, marker='o', markersize=3)

    # Mark major treaty events
    events = {1997: 'Ottawa\nConvention', 2008: 'CCM', 2013: 'ATT', 2017: 'TPNW'}
    for yr, label in events.items():
        if yr in term_df['year'].values:
            ax.axvline(yr, color='grey', linewidth=0.7, linestyle=':')
            ax.text(yr + 0.2, ax.get_ylim()[1]*0.92, label, fontsize=7.5, color='grey', rotation=90, va='top')

    ax.set_xlabel('Year'); ax.set_ylabel('Mean TF-IDF (normalized)')
    ax.set_title('Key Term Trajectories in Arms Control Discourse', fontweight='bold')
    ax.legend(ncol=2, fontsize=9)
    plt.tight_layout()
    plt.savefig(FIGURES / 'pres_fig4_term_trajectories.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('Term trajectory columns:', term_df.columns.tolist())
    print('Adjust column names above if needed.')

## Figure 5 — Moral Foundations Over Time

In [ ]:
moral_path = METRICS / 'moral_foundations_by_country_year.csv'
if moral_path.exists():
    moral_df = pd.read_csv(moral_path)
    mf_cols = ['care_harm', 'fairness', 'loyalty', 'authority', 'sanctity']
    mf_cols = [c for c in mf_cols if c in moral_df.columns]

    global_mf = moral_df.groupby('year')[mf_cols].mean()

    fig, ax = plt.subplots(figsize=(12, 5))
    colors_mf = ['#4575b4','#74add1','#fee090','#f46d43','#d73027']
    ax.stackplot(global_mf.index, [global_mf[c] for c in mf_cols],
                 labels=mf_cols, colors=colors_mf[:len(mf_cols)], alpha=0.8)
    ax.set_xlabel('Year')
    ax.set_ylabel('MFT keyword rate (per 100 words)')
    ax.set_title('Moral Foundations in Arms Control Rhetoric — Global Average', fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(FIGURES / 'pres_fig5_moral_foundations.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('Moral foundations file not found — run the full pipeline first.')

## Figure 6 — Summary Table for Slides

In [ ]:
# Key talking-point numbers for the presentation
gap_df = pd.read_csv(METRICS / 'rhetoric_action_gap.csv')
latest = gap_df[gap_df['year'] == gap_df['year'].max()]

if 'gap_category' in latest.columns:
    print('=== PRESENTATION KEY STATS ===')
    print(f"Year: {gap_df['year'].max()}")
    print(f"Countries analysed: {latest['country_code'].nunique()}")
    print(f"Mean rhetoric score: {latest['rhetoric_score'].mean():.3f}")
    print(f"Mean action score:   {latest['action_score'].mean():.3f}")
    print(f"Mean gap:            {latest['gap'].mean():+.3f}")
    print()
    print('Gap categories:')
    print(latest['gap_category'].value_counts().to_string())
    print()
    print('Biggest hypocrites (gap > 0.3):')
    hyp = latest[latest['gap_category'] == 'hypocrite'].sort_values('gap', ascending=False)
    print(hyp[['country_code', 'rhetoric_score', 'action_score', 'gap']].head(10).to_string(index=False))
    print()
    print('Biggest quiet good actors (gap < -0.3):')
    qga = latest[latest['gap_category'] == 'quiet_good_actor'].sort_values('gap')
    print(qga[['country_code', 'rhetoric_score', 'action_score', 'gap']].head(10).to_string(index=False))